# Build the final Maribyrnong catchment shapefile — fully self-contained

**This notebook produces, end-to-end and from a clean checkout, the simplified
10-gauge catchment shapefile that was uploaded to Google Earth Engine as
`projects/floodhubmaribyrnong/assets/ausvic_basin_shapes`.**

No external inputs are needed. Everything the notebook depends on (gauge
metadata, MWSTR download URL, output paths) is defined inline. No Colab Drive
mount. No reading from `gauges_ausvic.json`. No notebook 1, 2, 2a, or 2b.
Just run all cells in order and you get the final shapefile.

## What it produces

- `out/ausvic_basin_shapes.shp` (+ `.shx`, `.dbf`, `.prj`, `.cpg`) — the final
  simplified 10-gauge shapefile.
- `out/ausvic_basin_shapes.geojson` — same data, easy to inspect in QGIS / web.
- `out/ausvic_basin_shapes_diagnostics.csv` — per-gauge details.
- `out/comparison.png` — side-by-side render of raw and simplified polygons.

## Pipeline in one paragraph

Download the Melbourne Water Stream Network (MWSTR v1.3.1, 780 MB GeoPackage).
Read just the `subcs.site` + `subcs.nextds` columns and build a NetworkX directed
graph of the stream network. For each of the 10 gauges, look up its AWRC station
ID in MWSTR's `streamflow_gauges` table (7 of 10 succeed) or spatially snap to
the nearest stream reach (the remaining 3). Once we know the outlet site, use
`nx.ancestors` to enumerate every upstream sub-catchment, pull just those
polygons from SQLite directly (skipping the slow full-layer load), and union
them with `shapely.unary_union`. That gives the raw MWSTR-derived catchment per
gauge — ~14,000 raw vertices from the 5 m DEM (we apply a light 5 m cleanup during
the union; the diagnostics CSV breaks out mwstr_raw / 5m_clean / gee_simp). Earth Engine
can't handle that vertex density during Caravan Part-1, so we simplify with
Douglas-Peucker (~110 m tolerance) — removing redundant 5 m-spaced micro-vertices
that just trace the DEM grid, while keeping the catchment boundary identical to
the eye (max area drift: 0.23%). Save as a Caravan-format shapefile (single
combined `.shp`, only `gauge_id` attribute, EPSG:4326).

## Why these 10 gauges and not 12

Two earlier candidate gauges — `ausvic_230213` (Turritable Ck at Mt Macedon) and
`ausvic_230227` (Main Ck at Kerrie) — were dropped from the submission. They are
tiny upper-headwater monitoring stations (MWSTR gives them 5.3 km² and 4.7 km²)
with no agency cross-validation — neither MMBW 1986 nor Jacobs 2023 reports
them. The v2 submission used HydroBASINS L12 fallback areas (109.8 and 177.8 km²)
which were the L12 cell area not the actual catchment — a 20-38× overestimate.
Cleaner to drop them outright than re-submit with the corrected 20× smaller MWSTR
areas. See project README for the full rationale.

## Runtime

MWSTR download: ~3 minutes (cached). Graph build: ~3s. Outlet resolution: ~5s.
Per-gauge polygon union: 5–40s each. Simplification + save: ~10s. **Total: ~10
minutes including download, ~4 minutes if MWSTR is already cached.**


## Step 0 — Install dependencies


In [1]:
%pip install -q geopandas fiona shapely networkx matplotlib pyproj pyogrio


Note: you may need to restart the kernel to use updated packages.


## Step 1 — Imports


In [2]:
import sqlite3
import time
import urllib.request
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')  # comment out if running interactively in JupyterLab
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import pyogrio
import shapely
from shapely.geometry import Point
from shapely.wkb import loads as wkb_loads


## Step 2 — Inline configuration

All inputs defined here. Edit `OUT_DIR` for outputs, `WORK_DIR` for the cached
MWSTR download.

The 10 gauges are listed with AWRC station ID, (lat, lon) in WGS84, name, and
an `expected_area_km2` from the agency-validated source (MMBW 1986 or Jacobs
2023). We'll sanity-check MWSTR's derived areas against these.

MWSTR pinned to v1.3.1 (Walsh & Kunapo 2023, CC-BY-4.0). `VIC_CRS` is MWSTR's
native projected CRS — distances and areas are in metres there.


In [3]:
OUT_DIR  = Path('C:/Users/leela/FloodHubMaribyrnong/01_shapefile/outputs')
WORK_DIR = Path('C:/Users/leela/FloodHubMaribyrnong/01_shapefile/inputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

MWSTR_URL  = 'https://tools.thewerg.unimelb.edu.au/mwstr/data/mwstr_v1.3.1.gpkg'
MWSTR_GPKG = WORK_DIR / 'mwstr_v1.3.1.gpkg'
MWSTR_EXPECTED_SIZE = 817_868_800

VIC_CRS = 'EPSG:7855'  # GDA2020 / MGA Zone 55 — MWSTR's native

SIMPLIFY_TOLERANCE_DEG = 0.001  # ~110m at this latitude
SIMPLIFY_PRESERVE_TOPOLOGY = False

GAUGES = [
    # gauge_id, AWRC, lat, lon, name, expected area km² (None if no agency reference)
    ('ausvic_230119', '230119', -37.286090, 144.777750, 'Deep Creek at Doggetts Bridge Lancefield',     None),
    ('ausvic_230100', '230100', -37.410313, 144.902285, 'Deep Creek at Darraweit Guim',         500.0),
    ('ausvic_230107', '230107', -37.528500, 144.856000, 'Deep Creek at Konagaderra',     None),
    ('ausvic_230102', '230102', -37.631400, 144.801000, 'Deep Creek at Bulla',                  874.0),
    ('ausvic_230237', '230237', -37.677800, 144.805000, 'Maribyrnong River at Keilor North',    None),
    ('ausvic_230200', '230200', -37.727706, 144.836476, 'Maribyrnong River at Keilor',         1312.0),
    ('ausvic_230106', '230106', -37.765900, 144.895000, 'Maribyrnong River at Chifley Drive',   None),
    ('ausvic_230206', '230206', -37.475370, 144.572443, 'Jackson Creek at Gisborne',            None),
    ('ausvic_230202', '230202', -37.583217, 144.742036, 'Jackson Creek at Sunbury',             338.0),
    ('ausvic_230211', '230211', -37.466200, 144.744000, 'Bolinda Creek at Clarkefield',         None),
]
GAUGE_DF = pd.DataFrame(GAUGES, columns=['gauge_id','awrc','lat','lon','name','expected_area_km2'])
GAUGE_DF


,gauge_id,awrc,lat,lon,name,expected_area_km2
0,ausvic_230119,230119,-37.286090,144.777750,Deep Creek at Doggetts Bridge Lancefield,NaN
1,ausvic_230100,230100,-37.410313,144.902285,Deep Creek at Darraweit Guim,500.0
2,ausvic_230107,230107,-37.528500,144.856000,Deep Creek at Konagaderra,NaN
3,ausvic_230102,230102,-37.631400,144.801000,Deep Creek at Bulla,874.0
4,ausvic_230237,230237,-37.677800,144.805000,Maribyrnong River at Keilor North,NaN
5,ausvic_230200,230200,-37.727706,144.836476,Maribyrnong River at Keilor,1312.0
6,ausvic_230106,230106,-37.765900,144.895000,Maribyrnong River at Chifley Drive,NaN
7,ausvic_230206,230206,-37.475370,144.572443,Jackson Creek at Gisborne,NaN
8,ausvic_230202,230202,-37.583217,144.742036,Jackson Creek at Sunbury,338.0
9,ausvic_230211,230211,-37.466200,144.744000,Bolinda Creek at Clarkefield,NaN


## Step 3 — Download MWSTR (cached)

Skips if the file already exists and looks complete. Progress hook shows MB/total.


In [4]:
def _hook(blocks, block_size, total_size):
    done = blocks * block_size
    pct = 100.0 * done / total_size if total_size > 0 else 0
    print(f'\r  {done/1024/1024:7.1f} / {total_size/1024/1024:7.1f} MB ({pct:5.1f}%)', end='')

if not MWSTR_GPKG.exists() or MWSTR_GPKG.stat().st_size < MWSTR_EXPECTED_SIZE * 0.95:
    print(f'Downloading MWSTR v1.3.1 to {MWSTR_GPKG} ...')
    urllib.request.urlretrieve(MWSTR_URL, MWSTR_GPKG, reporthook=_hook)
    print('\nDone.')
else:
    print(f'MWSTR cached: {MWSTR_GPKG.stat().st_size/1024/1024:.1f} MB')


MWSTR cached: 780.0 MB


## Step 4 — Build the upstream-traversal graph

Open the GeoPackage as plain SQLite and `SELECT site, nextds, reach, carea_km2
FROM subcs`. No geometry — fast.

NetworkX DiGraph from `site → nextds`. Once built, `nx.ancestors(g, X)` returns
every sub-catchment that drains to X.


In [5]:
t0 = time.time()
try:                       # re-run-safe: drop any stale handle first
    con.close()
except Exception:
    pass
con = sqlite3.connect(str(MWSTR_GPKG))
rows = list(con.execute('SELECT site, nextds, reach, carea_km2 FROM subcs'))
print(f'  Loaded {len(rows):,} subcs rows in {time.time()-t0:.1f}s')

g_nx = nx.DiGraph()
reach_to_site = {}
site_to_carea = {}
for site, nextds, reach, carea in rows:
    site = int(site)
    if nextds is not None:
        g_nx.add_edge(site, int(nextds))
    else:
        g_nx.add_node(site)
    if reach:
        reach_to_site[reach] = site
    site_to_carea[site] = carea

print(f'  Graph: {g_nx.number_of_nodes():,} nodes  {g_nx.number_of_edges():,} edges  ({time.time()-t0:.1f}s)')


  Loaded 131,298 subcs rows in 2.5s
  Graph: 131,299 nodes  131,298 edges  (3.2s)


## Step 5 — Resolve each gauge to an MWSTR outlet site

Two strategies, tried in order:
1. **AWRC match**: try raw ID and trailing-letter variants ('A', 'B', 'C') in
   MWSTR's `streamflow_gauges` lookup. Works for the 7 Melbourne Water gauges.
2. **Spatial snap**: for the remaining 3 (Vic Water Hydstra), build Point from
   (lon, lat), reproject to EPSG:7855, `sjoin_nearest` against the streams
   layer in a 5 km bbox around the gauges. Snap distances should be < 200 m.


In [6]:
try:                       # reconnect if a previous full run closed the handle
    con.execute('SELECT 1')
except Exception:
    con = sqlite3.connect(str(MWSTR_GPKG))

gauge_rows = list(con.execute(
    'SELECT gauge_id_v, reach, sitecode, gauge_name FROM streamflow_gauges'))
awrc_to_reach = {r[0]: (r[1], r[2], r[3]) for r in gauge_rows if r[0]}
print(f'MWSTR has {len(awrc_to_reach):,} gauge \u2192 reach mappings\n')

outlets = {}
to_snap = []

for _, g in GAUGE_DF.iterrows():
    gid = g['gauge_id']
    candidates = [g['awrc'], g['awrc'] + 'A', g['awrc'] + 'B', g['awrc'] + 'C']
    found = next((c for c in candidates if c in awrc_to_reach), None)
    if found:
        reach, sitecode, _ = awrc_to_reach[found]
        site = reach_to_site.get(reach)
        if site is not None:
            outlets[gid] = {'site': site, 'reach': reach, 'sitecode': sitecode, 'method': 'awrc_match'}
            print(f'  \u2713 AWRC  {gid:<22} awrc={found:<10} reach={reach}')
            continue
    to_snap.append(g.to_dict())
    print(f'  ?      {gid:<22} \u2192 will spatially snap')

if to_snap:
    pts = gpd.GeoDataFrame(
        to_snap,
        geometry=[Point(g['lon'], g['lat']) for g in to_snap],
        crs='EPSG:4326',
    ).to_crs(VIC_CRS)
    minx, miny = pts.total_bounds[0] - 5000, pts.total_bounds[1] - 5000
    maxx, maxy = pts.total_bounds[2] + 5000, pts.total_bounds[3] + 5000
    streams = gpd.read_file(MWSTR_GPKG, layer='streams',
                            bbox=(minx, miny, maxx, maxy), engine='fiona')
    snap = gpd.sjoin_nearest(pts, streams[['site','reach','geometry']],
                             how='left', distance_col='snap_m')
    for _, r in snap.iterrows():
        outlets[r['gauge_id']] = {
            'site':   int(r['site']),
            'reach':  r['reach'],
            'method': f'spatial_snap_{r["snap_m"]:.0f}m',
        }
        print(f'  \u2713 SNAP  {r["gauge_id"]:<22} site={int(r["site"]):>7d}  dist={r["snap_m"]:.0f}m')

print(f'\nResolved {len(outlets)} / {len(GAUGE_DF)} outlets')


MWSTR has 165 gauge → reach mappings

  ✓ AWRC  ausvic_230119          awrc=230119A    reach=DPW_21930
  ✓ AWRC  ausvic_230100          awrc=230100A    reach=DPW_48389
  ✓ AWRC  ausvic_230107          awrc=230107A    reach=DPW_62074
  ✓ AWRC  ausvic_230102          awrc=230102A    reach=DPW_86083
  ✓ AWRC  ausvic_230237          awrc=230237A    reach=MRB_127992
  ?      ausvic_230200          → will spatially snap
  ✓ AWRC  ausvic_230106          awrc=230106A    reach=MRB_138648
  ?      ausvic_230206          → will spatially snap
  ?      ausvic_230202          → will spatially snap
  ✓ AWRC  ausvic_230211          awrc=230211A    reach=BOL_9615
  ✓ SNAP  ausvic_230200          site=  55183  dist=8m
  ✓ SNAP  ausvic_230206          site= 360155  dist=20m
  ✓ SNAP  ausvic_230202          site=  27785  dist=1m

Resolved 10 / 10 outlets


## Step 6 — Union upstream sub-catchments per gauge

Two performance tricks: direct SQLite `SELECT geom FROM subcs WHERE site IN (...)`
(faster than `gpd.read_file(layer='subcs')`), and manual GeoPackage WKB decoding.

Order: union first, **then** simplify (simplifying each polygon independently
before union can shift shared boundaries and create slivers).

Cross-check: MWSTR area vs the agency-validated `expected_area_km2`.


In [7]:
try:                       # reconnect if a previous full run closed the handle
    con.execute('SELECT 1')
except Exception:
    con = sqlite3.connect(str(MWSTR_GPKG))

def gpkg_blob_to_wkb(blob):
    flags = blob[3]
    env_bits = (flags >> 1) & 0x07
    env_sizes = {0: 0, 1: 32, 2: 48, 3: 48, 4: 64}
    header_len = 8 + env_sizes.get(env_bits, 0)
    return blob[header_len:]

def union_one_gauge(outlet_site):
    upstream = {outlet_site}
    if outlet_site in g_nx:
        upstream |= nx.ancestors(g_nx, outlet_site)
    sites = list(upstream)
    polys = []
    CHUNK = 500
    for i in range(0, len(sites), CHUNK):
        chunk = sites[i:i + CHUNK]
        q = f"SELECT geom FROM subcs WHERE site IN ({','.join(['?'] * len(chunk))})"
        for (blob,) in con.execute(q, chunk):
            if blob is None:
                continue
            try:
                polys.append(wkb_loads(gpkg_blob_to_wkb(blob)))
            except Exception:
                pass
    unioned = shapely.unary_union(polys)
    raw_vtx = (len(unioned.exterior.coords) if unioned.geom_type == 'Polygon'
               else sum(len(p.exterior.coords) for p in unioned.geoms))
    unioned = unioned.simplify(5, preserve_topology=True)
    return unioned, len(polys), raw_vtx

raw_results = []
for _, gm in GAUGE_DF.iterrows():
    gid = gm['gauge_id']
    if gid not in outlets:
        print(f'  \u2717 {gid}: no outlet — skip'); continue
    t = time.time()
    site = outlets[gid]['site']
    poly, n, raw_vtx = union_one_gauge(site)
    area = poly.area / 1e6
    expected = gm['expected_area_km2']
    diff_str = 'n/a'
    if pd.notna(expected):
        diff = abs(area - expected) / expected * 100
        diff_str = f'{diff:.1f}%'
    raw_results.append({
        'gauge_id': gid, 'name': gm['name'],
        'outlet_site': site, 'method': outlets[gid]['method'],
        'n_subcatchments': n, 'verts_mwstr_raw': raw_vtx, 'area_mwstr_km2': round(area, 1),
        'expected_km2': expected, 'area_diff_%': diff_str,
        'geometry': poly,
    })
    print(f'  {gid:<22} n={n:>5d}  area={area:>8.1f} km\u00b2  expected={expected if expected else "n/a":<8}  diff={diff_str}  ({time.time()-t:.1f}s)')

raw_gdf = gpd.GeoDataFrame(raw_results, geometry='geometry', crs=VIC_CRS)
print(f'\nDerived {len(raw_gdf)} raw polygons')


  ausvic_230119          n= 2296  area=   219.3 km²  expected=nan       diff=n/a  (7.1s)
  ausvic_230100          n= 5771  area=   483.9 km²  expected=500.0     diff=3.2%  (16.1s)
  ausvic_230107          n= 6982  area=   620.7 km²  expected=nan       diff=n/a  (19.0s)
  ausvic_230102          n= 8633  area=   860.8 km²  expected=874.0     diff=1.5%  (25.4s)
  ausvic_230237          n=12287  area=  1279.9 km²  expected=nan       diff=n/a  (37.2s)
  ausvic_230200          n=12435  area=  1306.4 km²  expected=1312.0    diff=0.4%  (46.6s)
  ausvic_230106          n=12696  area=  1386.5 km²  expected=nan       diff=n/a  (45.8s)
  ausvic_230206          n= 1330  area=    91.2 km²  expected=nan       diff=n/a  (3.2s)
  ausvic_230202          n= 3195  area=   342.7 km²  expected=338.0     diff=1.4%  (9.4s)
  ausvic_230211          n=  682  area=    96.2 km²  expected=nan       diff=n/a  (2.6s)

Derived 10 raw polygons


## Step 7 — Quality gate on raw polygons

Two checks before we simplify: polygon uniqueness, and areas-vs-expected within
5 %. Anything > 5 % would suggest the outlet resolution went to the wrong reach.


In [8]:
# 7a uniqueness
hashes = raw_gdf.geometry.apply(lambda p: hash(p.wkb))
assert not hashes.duplicated().any(), 'Duplicate polygons'
print(f'\u2713 All {len(raw_gdf)} polygons are unique')

# 7b area sanity
for _, r in raw_gdf.iterrows():
    if pd.notna(r['expected_km2']):
        d = abs(r['area_mwstr_km2'] - r['expected_km2']) / r['expected_km2'] * 100
        assert d < 5, f'{r["gauge_id"]} area drift {d:.1f}% > 5%'
print('\u2713 All gauges within 5% of agency-validated areas')

# 7c topological nesting: if outlet B is upstream of outlet A in the stream graph,
# then gauge A's catchment must contain B's and be at least as large. Validates the
# tracing for ALL gauges, including the 6 with no agency area to check against.
_site = {gid: outlets[gid]['site'] for gid in raw_gdf['gauge_id']}
_poly = {r['gauge_id']: r.geometry for _, r in raw_gdf.iterrows()}
_pairs = 0
for _a in _site:
    _anc = (nx.ancestors(g_nx, _site[_a]) | {_site[_a]}) if _site[_a] in g_nx else {_site[_a]}
    for _b in _site:
        if _a == _b or _site[_b] not in _anc:
            continue
        _frac = _poly[_a].intersection(_poly[_b]).area / _poly[_b].area
        assert _frac > 0.97, f'{_b} upstream of {_a} but only {_frac:.1%} contained'
        assert _poly[_a].area >= _poly[_b].area * 0.98, f'{_a} smaller than upstream {_b}'
        _pairs += 1
print(f'✓ Topological nesting consistent across {_pairs} upstream/downstream pairs')

# 7d independent area cross-check vs MWSTR's pre-computed cumulative area (carea_km2)
print()
print(f'  {"gauge_id":<22} {"geom_km2":>9} {"carea_km2":>10} {"diff":>7}')
for _, r in raw_gdf.iterrows():
    _ca = site_to_carea.get(outlets[r['gauge_id']]['site'])
    if _ca:
        _d = (r.geometry.area / 1e6 - _ca) / _ca * 100
        _flag = '' if abs(_d) < 5 else '  <-- check'
        print(f'  {r["gauge_id"]:<22} {r.geometry.area/1e6:>9.1f} {_ca:>10.1f} {_d:>+6.1f}%{_flag}')


✓ All 10 polygons are unique
✓ All gauges within 5% of agency-validated areas
✓ Topological nesting consistent across 32 upstream/downstream pairs

  gauge_id                geom_km2  carea_km2    diff
  ausvic_230119              219.3      219.3   +0.0%
  ausvic_230100              483.9      483.9   +0.0%
  ausvic_230107              620.7      620.7   +0.0%
  ausvic_230102              860.8      860.8   +0.0%
  ausvic_230237             1279.9     1279.9   +0.0%
  ausvic_230200             1306.4     1306.4   +0.0%
  ausvic_230106             1386.5     1386.5   +0.0%
  ausvic_230206               91.2       91.2   +0.0%
  ausvic_230202              342.7      342.7   +0.0%
  ausvic_230211               96.2       96.2   -0.0%


## Step 8 — Reproject to WGS84 and simplify

Reproject to EPSG:4326 (Caravan + GEE convention). Apply Douglas-Peucker at
0.001° (~110m). `preserve_topology=False` uses the fast Hershberger variant; we
validate topology after.


In [9]:
raw_wgs = raw_gdf[['gauge_id', 'geometry']].to_crs('EPSG:4326').copy()

t0 = time.time()
simp_wgs = raw_wgs.copy()
simp_wgs.geometry = simp_wgs.geometry.simplify(
    SIMPLIFY_TOLERANCE_DEG, preserve_topology=SIMPLIFY_PRESERVE_TOPOLOGY)
print(f'Simplified {len(simp_wgs)} polygons in {time.time()-t0:.2f}s')

def count_verts(g):
    if g.geom_type == 'Polygon': return len(g.exterior.coords)
    if g.geom_type == 'MultiPolygon': return sum(len(p.exterior.coords) for p in g.geoms)
    return 0

# three honest stages: MWSTR raw (pre-cleanup) -> 5 m cleanup -> 110 m DP for GEE
v_mwstr = raw_gdf['verts_mwstr_raw'].tolist()
v_raw  = raw_wgs.geometry.apply(count_verts)
v_simp = simp_wgs.geometry.apply(count_verts)
print()
print(f'  {"gauge_id":<22} {"mwstr_raw":>10} {"5m_clean":>10} {"gee_simp":>10} {"reduction":>11}')
print('  ' + '-' * 67)
for gid, vm, vr, vs in zip(raw_wgs['gauge_id'], v_mwstr, v_raw, v_simp):
    print(f'  {gid:<22} {vm:>10,} {vr:>10,} {vs:>10,} {100*(1-vs/vm):>10.1f}%')
print('  ' + '-' * 67)
print(f'  {"TOTAL":<22} {int(sum(v_mwstr)):>10,} {int(v_raw.sum()):>10,} {int(v_simp.sum()):>10,} {100*(1-v_simp.sum()/sum(v_mwstr)):>10.1f}%')


Simplified 10 polygons in 0.07s

  gauge_id                mwstr_raw   5m_clean   gee_simp   reduction
  -------------------------------------------------------------------
  ausvic_230119              11,404      2,890        140       98.8%
  ausvic_230100              18,933      4,983        214       98.9%
  ausvic_230107              22,133      5,766        269       98.8%
  ausvic_230102              24,403      6,241        279       98.9%
  ausvic_230237              34,346      8,903        415       98.8%
  ausvic_230200              37,243      9,735        451       98.8%
  ausvic_230106              41,916     11,359        524       98.7%
  ausvic_230206               8,981      2,369        110       98.8%
  ausvic_230202              17,737      4,797        203       98.9%
  ausvic_230211               6,115      1,558         66       98.9%
  -------------------------------------------------------------------
  TOTAL                     223,211     58,601      2,671

## Step 9 — Validate the simplification

Area drift (< 1 %) and topology validity. Both must pass before we save.


In [10]:
raw_areas = {r['gauge_id']: r.geometry.area for _, r in raw_wgs.iterrows()}
max_drift = 0.0
all_valid = True
for _, r in simp_wgs.iterrows():
    ra = raw_areas[r['gauge_id']]
    sa = r.geometry.area
    drift = abs(sa - ra) / ra * 100 if ra else 0
    max_drift = max(max_drift, drift)
    all_valid = all_valid and r.geometry.is_valid

print(f'Max area drift: {max_drift:.3f}%')
print(f'All polygons topologically valid: {all_valid}')
assert all_valid, 'Invalid polygon!'
assert max_drift < 1.0, f'Area drift {max_drift:.2f}% > 1%'
print('\u2713 simplification valid')


Max area drift: 0.195%
All polygons topologically valid: True
✓ simplification valid


## Step 10 — Save the Caravan-format shapefile

Single combined `.shp`, only `gauge_id` column, EPSG:4326, all 5 siblings.
Plus a GeoJSON copy and a diagnostics CSV.


In [11]:
shp_path  = OUT_DIR / 'ausvic_basin_shapes.shp'
gj_path   = OUT_DIR / 'ausvic_basin_shapes.geojson'
diag_path = OUT_DIR / 'ausvic_basin_shapes_diagnostics.csv'

out_shp = simp_wgs[['gauge_id', 'geometry']].copy()
pyogrio.write_dataframe(out_shp, shp_path, driver='ESRI Shapefile')
out_shp.to_file(gj_path, driver='GeoJSON')

diag = raw_gdf.drop(columns=['geometry']).copy()
diag['verts_5m_clean'] = list(v_raw)
diag['verts_simp'] = list(v_simp)
diag['vert_reduction_%'] = [round(100*(1-vs/vm), 1) for vm, vs in zip(diag['verts_mwstr_raw'], v_simp)]
diag.to_csv(diag_path, index=False, encoding='utf-8-sig')

print('Wrote:')
for p in [shp_path.with_suffix('.shp'), shp_path.with_suffix('.shx'),
          shp_path.with_suffix('.dbf'), shp_path.with_suffix('.prj'),
          shp_path.with_suffix('.cpg'), gj_path, diag_path]:
    if p.is_file():
        print(f'  \u2713 {p}  ({p.stat().st_size:,} bytes)')
    else:
        print(f'  \u2717 {p}  (missing)')


Wrote:
  ✓ C:\Users\leela\FloodHubMaribyrnong\01_shapefile\outputs\ausvic_basin_shapes.shp  (43,396 bytes)
  ✓ C:\Users\leela\FloodHubMaribyrnong\01_shapefile\outputs\ausvic_basin_shapes.shx  (180 bytes)
  ✓ C:\Users\leela\FloodHubMaribyrnong\01_shapefile\outputs\ausvic_basin_shapes.dbf  (876 bytes)
  ✓ C:\Users\leela\FloodHubMaribyrnong\01_shapefile\outputs\ausvic_basin_shapes.prj  (145 bytes)
  ✓ C:\Users\leela\FloodHubMaribyrnong\01_shapefile\outputs\ausvic_basin_shapes.cpg  (5 bytes)
  ✓ C:\Users\leela\FloodHubMaribyrnong\01_shapefile\outputs\ausvic_basin_shapes.geojson  (123,723 bytes)
  ✓ C:\Users\leela\FloodHubMaribyrnong\01_shapefile\outputs\ausvic_basin_shapes_diagnostics.csv  (1,166 bytes)


## Step 11 — Proof: every gauge sits at its catchment outlet

A reviewer note on the earlier submission was that gauge markers sometimes fell in the **middle**
of their polygons rather than at the outlet — which would mean the polygon didn't represent the
true upstream area. The MWSTR construction here cannot produce that, and this step proves it.

**Correct by construction.** Each polygon is the union of the gauge's own outlet sub-catchment and
*every* sub-catchment upstream of it (`nx.ancestors()` on the MWSTR flow network). The polygon is
therefore, by definition, exactly the area that drains *through* the gauge — the gauge is the
outlet, never an interior point.

Three checks below confirm it for all 10 gauges:
1. **Topology** — the sub-catchment immediately downstream of each gauge lies *outside* the polygon,
   so the catchment has a single pour point: the gauge.
2. **Geometry** — each gauge lies on its catchment boundary (tens to a few hundred metres from the
   edge) yet 6–38 km from the centroid: hard against the downstream rim, not in the middle.
3. **Visual** — the panels show each gauge (red) at the downstream tip of its own catchment.

**On the apparent "middle" effect:** these are *nested* catchments (Konagaderra ⊂ Bulla ⊂ …, and
Keilor Nth ⊂ Keilor ⊂ Chifley Dr). On a single combined map an upstream gauge naturally falls
*inside* a larger downstream polygon — correct hydrology, not a misplacement. Each gauge is at the
outlet of **its own** catchment, as the one-gauge-per-panel plot below shows.

**Note — 230106 (Chifley Drive) prints `in poly: False`.** Its gauge point sits ~16 m *outside* the simplified boundary: Douglas-Peucker trimmed the outlet rim a fraction inside the gauge at the tidal river mouth. This is geometrically negligible (16 m on a 1,386 km² catchment, well within the ~110 m simplification tolerance) and changes neither the catchment area nor the derived forcings (the gauge point is metadata, not part of the polygon). The topology column confirms it remains the catchment's single outlet.

In [12]:
## Step 11 — proof that each gauge is at its catchment outlet
import matplotlib.pyplot as plt
from shapely.geometry import Point

_p = gpd.read_file(OUT_DIR / 'ausvic_basin_shapes.geojson')
if _p.crs is None: _p = _p.set_crs('EPSG:4326')
_p = _p.to_crs('EPSG:4326').set_index('gauge_id')
_pm = _p.to_crs(VIC_CRS)                                   # metres, for distances
have_net = 'g_nx' in globals() and 'outlets' in globals()  # extra topology check on a full run

print('Outlet proof - every gauge is the single pour point of its catchment')
print('=' * 74)
hdr = f'{"gauge":<16}{"in poly":>8}{"to edge":>10}{"to centroid":>14}'
if have_net: hdr += f'{"drains via gauge":>18}'
print(hdr)
rows = []
for _, g in GAUGE_DF.iterrows():
    gid = g['gauge_id']
    poly_m = _pm.loc[gid, 'geometry']
    pt_m = gpd.GeoSeries([Point(g['lon'], g['lat'])], crs='EPSG:4326').to_crs(VIC_CRS).iloc[0]
    inside = poly_m.contains(pt_m)
    d_edge = pt_m.distance(poly_m.boundary)
    d_cen  = pt_m.distance(poly_m.centroid)
    topo = None
    if have_net:
        site = outlets[gid]['site']
        cat  = {site} | nx.ancestors(g_nx, site)
        topo = next(iter(g_nx.successors(site)), None) not in cat
    print(f'{gid:<16}{str(inside):>8}{d_edge:>8.0f} m{d_cen/1000:>11.1f} km'
          + (f'{("yes" if topo else "NO"):>18}' if have_net else ''))
    rows.append((d_edge, d_cen, topo))

assert all(de < 0.2 * dc for de, dc, _ in rows), 'a gauge is not near its catchment edge'
if have_net:
    assert all(t for *_, t in rows), 'a catchment drains somewhere other than its gauge'
print('\nAll 10 lie on the catchment boundary (tens-hundreds of m from the edge, km from the')
print('centroid) -> each gauge is at the outlet, not in the middle.')

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, (_, g) in zip(axes.ravel(), GAUGE_DF.iterrows()):
    poly = _p.loc[g['gauge_id'], 'geometry']
    gpd.GeoSeries([poly], crs='EPSG:4326').plot(ax=ax, fc='#cfe8ff', ec='#1f6fb2', lw=1.2)
    c = poly.centroid
    ax.plot(c.x, c.y, 'x', color='gray', ms=9, mew=2, label='centroid')
    ax.plot(g['lon'], g['lat'], 'o', color='red', ms=9, mec='k', label='gauge (outlet)')
    ax.set_title(g['name'], fontsize=8.5); ax.set_xticks([]); ax.set_yticks([])
axes.ravel()[0].legend(loc='upper left', fontsize=8)
fig.suptitle('Each gauge (red) sits at the outlet - the downstream tip - of its own catchment, far from the centroid (grey x)', fontsize=13)
fig.tight_layout()
fig.savefig(OUT_DIR / 'outlet_proof.png', dpi=110, bbox_inches='tight')
print(f'Saved {OUT_DIR / "outlet_proof.png"}')
plt.show()

Outlet proof - every gauge is the single pour point of its catchment
gauge            in poly   to edge   to centroid  drains via gauge
ausvic_230119       True      63 m        8.5 km               yes
ausvic_230100       True     107 m       15.9 km               yes
ausvic_230107       True     109 m       20.7 km               yes
ausvic_230102       True     303 m       27.5 km               yes
ausvic_230237       True     134 m       29.1 km               yes
ausvic_230200       True      69 m       34.5 km               yes
ausvic_230106      False      16 m       38.2 km               yes
ausvic_230206       True      24 m        6.3 km               yes
ausvic_230202       True       6 m       17.0 km               yes
ausvic_230211       True     153 m        8.5 km               yes

All 10 lie on the catchment boundary (tens-hundreds of m from the edge, km from the
centroid) -> each gauge is at the outlet, not in the middle.
Saved C:\Users\leela\FloodHubMaribyrnong\01_shap

C:\Users\leela\AppData\Local\Temp\ipykernel_21196\1092667653.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 11 — Final verification

Read the file back, run Caravan's 7 checks. All must pass.


In [13]:
rt = pyogrio.read_dataframe(shp_path)
wkb_hashes = [g.wkb_hex[:32] for g in rt.geometry]
checks = {
    f'has {len(GAUGE_DF)} features':       len(rt) == len(GAUGE_DF),
    'only gauge_id column':                list(rt.columns) == ['gauge_id', 'geometry'],
    'CRS = EPSG:4326':                     rt.crs.to_string().upper() == 'EPSG:4326',
    'all polygons unique':                 len(set(wkb_hashes)) == len(rt),
    'all topologically valid':             all(g.is_valid for g in rt.geometry),
    'no gauge_id ausvic_230213':           'ausvic_230213' not in rt['gauge_id'].tolist(),
    'no gauge_id ausvic_230227':           'ausvic_230227' not in rt['gauge_id'].tolist(),
}
CHK_MARK, CROSS_MARK = "\u2713", "\u2717"
for name, ok in checks.items():
    print(f'  {CHK_MARK if ok else CROSS_MARK} {name}')
assert all(checks.values())
print('\nReady to upload to Google Earth Engine.')


  ✓ has 10 features
  ✓ only gauge_id column
  ✓ CRS = EPSG:4326
  ✓ all polygons unique
  ✓ all topologically valid
  ✓ no gauge_id ausvic_230213
  ✓ no gauge_id ausvic_230227

Ready to upload to Google Earth Engine.


## Step 12 — Side-by-side render

Visual sanity check. Optional but pretty.


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for _, r in raw_wgs.iterrows():
    g = r.geometry
    polys = [g] if g.geom_type == 'Polygon' else list(g.geoms)
    for p in polys:
        x, y = p.exterior.xy
        axes[0].fill(list(x), list(y), color='#d62728', alpha=0.15)
        axes[0].plot(list(x), list(y), color='#d62728', lw=1.0)
axes[0].set_aspect('equal')
axes[0].set_title(f"MWSTR catchments (5 m-cleaned)\n{int(v_raw.sum()):,} verts  |  raw {int(sum(v_mwstr)):,}")
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

for _, r in simp_wgs.iterrows():
    g = r.geometry
    polys = [g] if g.geom_type == 'Polygon' else list(g.geoms)
    for p in polys:
        x, y = p.exterior.xy
        axes[1].fill(list(x), list(y), color='#2ca02c', alpha=0.15)
        axes[1].plot(list(x), list(y), color='#2ca02c', lw=1.0)
axes[1].set_aspect('equal')
axes[1].set_title(f"Simplified (~110m DP)\n{int(v_simp.sum()):,} total vertices")
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

plt.suptitle('Final 10-gauge Maribyrnong catchments')
plt.tight_layout()
plt.savefig(OUT_DIR / 'comparison.png', dpi=110, bbox_inches='tight')
print(f'Saved {OUT_DIR / "comparison.png"}')

con.close()   # tidy up the SQLite handle (re-run from Step 4 if you need it again)


Saved C:\Users\leela\FloodHubMaribyrnong\01_shapefile\outputs\comparison.png


## What to do next

Upload the 5 shapefile siblings from `out/` to Earth Engine:

1. Open https://code.earthengine.google.com/
2. Left panel \u2192 **Assets** \u2192 **NEW** \u2192 **Shape files**
3. **SELECT** \u2192 choose all 5 files
4. **Asset ID**: `ausvic_basin_shapes`
5. **UPLOAD** (~30s)

Then run Caravan Part-1 to generate HydroATLAS attributes + ERA5-Land forcings,
followed by Part-2 to assemble the final Caravan bundle.
